<a href="https://colab.research.google.com/github/ncinsli/CLIP-classification-experiments/blob/main/4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 4

*Change text to improve results. Try to take mean from embeddings*

In [ ]:
import gc
import torch
import torchvision
import numpy as np
import transformers
from PIL import Image
from tqdm.notebook import tqdm
import torch.nn.functional as F
from collections import Counter
import matplotlib.pyplot as plt
from torchvision import transforms
from sklearn import metrics, preprocessing
from transformers import CLIPModel, CLIPProcessor, CLIPTokenizer

In [ ]:
BATCH_SIZE = 128
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
model.eval()
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

In [ ]:
imagenette_data = torchvision.datasets.Imagenette('imagenette/', download=True)
data_loader = torch.utils.data.DataLoader(imagenette_data,
                                          batch_size=BATCH_SIZE,
                                          shuffle=False,
                                          num_workers=2,
                                          collate_fn=lambda b: ([i[0] for i in b], [i[1] for i in b]))

## Image embeddings retreival

This cell calculates image embeddings so for the further experiments we only need to rerun text embeddings retreival.

In [ ]:
def get_image_embeddings(model, data_loader):
  truth = []
  image_embeddings = torch.tensor([]).to(device)

  for batch, t in tqdm(data_loader):
    with torch.inference_mode():
      img_inputs = processor(images=batch, padding=True, return_tensors='pt').to(device)
      image_emb = model.get_image_features(**img_inputs).pooler_output.detach().to(device)
      image_emb = F.normalize(image_emb, 2, dim=1)

      image_embeddings = torch.cat((image_embeddings, image_emb), dim=0)
      truth += t
  return (image_embeddings, truth)

In [ ]:
image_embeddings, truth = get_image_embeddings(model, data_loader)

## Text embeddings retreival and prediction forming

In [ ]:
def get_text_embeddings(model, tokenizer, classes):
  txt_inputs = tokenizer(text=classes_for_clip, padding=True, return_tensors="pt").to(device)
  text_features = model.get_text_features(**txt_inputs)
  text_emb = text_features.pooler_output.detach().to(device)
  return F.normalize(text_emb, 2, dim=1)

In [ ]:
# classes_for_clip = [f'{', '.join(i)}' for i in imagenette_data.classes]
classes_for_clip = ['An underwater tench', 'A joyful english springer', 'An old cassette player', 'A sharp chainsaw', 'A majestic church', 'A metallic french horn', 'A useful garbage truck', 'A white gas pump', 'A round golf ball', 'A bright parachute']
text_embeddings = get_text_embeddings(model, tokenizer, classes_for_clip)

print(classes_for_clip)

predictions = []
logits = (image_embeddings @ text_embeddings.T) * np.exp(model.logit_scale.item())
predicted_cat = logits.argmax(dim=1).to('cpu')
predictions = predicted_cat.tolist()

In [ ]:
def display_metrics(truth, predictions):
  print(f'Accuracy    {metrics.accuracy_score(truth, predictions)}')
  print(f'Precision   {metrics.precision_score(truth, predictions, average='macro')}')
  print(f'Recall      {metrics.recall_score(truth, predictions, average='macro')}')
  print(f'F1          {metrics.f1_score(truth, predictions, average='macro')}')
  print()

  freqs = Counter(predictions)
  true_freqs = Counter(truth)

  fig, ax = plt.subplots(1, 2)
  fig.set_figwidth(15)
  fig.suptitle('Imagenette class sizes')

  ax[0].bar(freqs.keys(), freqs.values())
  ax[0].set_xlabel('Predicted distribution')
  ax[0].set_xticks(range(10))

  ax[1].bar(true_freqs.keys(), true_freqs.values())
  ax[1].set_xlabel('True distribution')
  ax[1].set_xticks(range(10))

display_metrics(truth, predictions)

## Prompt engineering thoughts

Reformulating categories just slightly changes metrics with pertubation of about 0.002. The bare category names achieve 0.988 at all metrics, while 'A picture of {classname}' approach achieves only 0.985.

It was interesting to notice that '*{classname} wonderful picture*' made model predict tench more oftenly. Howewer, due to the trend instability, we consider that to be a coincidence

## Embedding averaging

In [ ]:
classes_options = [[f'{j} {i[0]}' for i in imagenette_data.classes] for j in ('A picture of', 'A photo of')]
option_count = len(classes_options[0])
mean_text_embeddings = torch.zeros((option_count, 10, 512)).to(device)

for i, class_option in enumerate(classes_options):
  mean_text_embeddings[i] = get_text_embeddings(model, tokenizer, class_option)

mean_text_embeddings = mean_text_embeddings.mean(dim=0)
mean_text_embeddings = F.normalize(mean_text_embeddings, 2, dim=1)

logits = (image_embeddings @ mean_text_embeddings.T) * np.exp(model.logit_scale.item())
predicted_cat = logits.argmax(dim=1).to('cpu')
predictions = predicted_cat.tolist()

display_metrics(truth, predictions)